In [ ]:
import sys, os
import h5py
import numpy as np

from skimage.measure import block_reduce

from helper_functions import write_text
from matplotlib import pyplot as plt

if os.path.exists('/home/ayyerkar/.local/dragonfly/utils/py_src/'):
    sys.path.append('/home/ayyerkar/.local/dragonfly/utils/py_src/')
    print('Appended Dragonfly!') 
    import detector
else: 
    print('Dragonfly not found, please install Dragonfly/provide different path!') 
    
plotFig = True

writeDetMask = True
saveBin = True

<h2> Writing (downsampled) detector mask </h2>

In [ ]:
mask_name = 'emc/make_detector/agipd_detector_mask.h5'
mask_type = 'agipd'
with h5py.File(mask_name, mode='r') as agipd:
    d_mask = agipd['mask'][:]
write_text(f'Shape of AGIPD mask: {d_mask.shape}\n')
write_text('Cropping AGIPD to square array!\n')

cy_agipd, cx_agipd = d_mask.shape[0]//2, d_mask.shape[1]//2
d_mask = d_mask[cy_agipd-cx_agipd:cy_agipd+cx_agipd,:]
cy_agipd, cx_agipd = d_mask.shape[0]//2, d_mask.shape[1]//2
write_text(f'Shape of cropped AGIPD mask: (%s, %s)\n' % (d_mask.shape[0],d_mask.shape[1]))
cx = cx_agipd
cy = cy_agipd
dimY, dimX = d_mask.shape

downsample_factor = 4

if downsample_factor == 1:
    print('No downsampling!')
    d_mask_ds = d_mask
    plt.figure(dpi=100)
    plt.imshow(d_mask_ds, cmap='gray')
    plt.plot(cx, cy, 'rx')
    cb = plt.colorbar();

    cc = d_mask_ds.shape[0]//2

    d_mask_ds = d_mask_ds[cc-128:cc+128,cc-128:cc+128]
    plt.figure(dpi=100)
    plt.imshow(d_mask_ds, cmap='gray')
    plt.plot(128, 128, 'rx')
    cb = plt.colorbar();
    
else:
    while dimX%downsample_factor!=0 or dimY%downsample_factor!=0:
        print('Downsampling would result in non-integer array size!')
        print('Choose another downsampling factor!!!')
        downsample_factor = int(input(prompt='Downsampling factor:'))
    
    d_mask_float = d_mask.astype(float)
    d_mask_float[d_mask==False] = np.nan
    
    d_mask_ds = block_reduce(d_mask_float, block_size=downsample_factor, func=np.nansum)
    write_text(f'Number of good pixels in each super pixel: {np.unique(d_mask_ds)}\n')
    d_mask_ds = (d_mask_ds >= 16.) # 56. for 12x downsampling, 15. for 7x downsampling, 2. for 6x downsampling, and 1. for 4x downsampling and 16. for strict 4x downsampling (test)
    
    write_text(f'Shape of downsampled detector mask: {d_mask_ds.shape}\n\n')
    cy_ds, cx_ds = d_mask_ds.shape[0]//2, d_mask_ds.shape[1]//2

    write_text(f'Fraction of good pixels before and after downsampling: {(d_mask.sum())/(d_mask.shape[0]*d_mask.shape[1])} and {(d_mask_ds.sum())/(d_mask_ds.shape[0]*d_mask_ds.shape[1])}')
    
    if plotFig: 
        plt.figure(dpi=100)
        plt.imshow(d_mask_float, cmap='gray')
        plt.plot(cx, cy, 'rx');
        cb = plt.colorbar()
     
        plt.figure(dpi=100)
        plt.imshow(d_mask_ds, cmap='gray')
        plt.plot(cx_ds, cy_ds, 'rx')
        cb = plt.colorbar();
    
if writeDetMask:
    with h5py.File(f'full_mask_{mask_type}_ds_{downsample_factor}_strict_mask.h5','w') as msk: 
        msk.create_dataset('mask',data=d_mask_ds)

To proceed with the EMC using `make_detector` I first need to convert my detector mask to a binary file with the correct data type. 
Make sure to use `u1` and not use `np.uint8`. 

In [ ]:
if saveBin:
    d_mask_ds_u1 = d_mask_ds.copy().astype('u1')
    mask_good = (d_mask_ds_u1==True)
    mask_bad = (d_mask_ds_u1==False)
    d_mask_ds_u1[mask_good] = 0
    d_mask_ds_u1[mask_bad] = 2

    d_mask_ds_u1.astype('u1').tofile(f'mask_{mask_type}_ds_{downsample_factor}_strict_mask.byt')